# sPHENIX TPC 3D ion-backflow field generator

This notebook builds a realistic side/sector/module-dependent ion charge distribution, optionally estimates and fits its radial power law, performs a memory-bounded Green-function convolution, and writes local Cartesian field maps for `PHGarfield`.

In [1]:
import re, json, time
from pathlib import Path
import numpy as np
import ROOT
from scipy import optimize, special, integrate
import matplotlib.pyplot as plt

ROOT.gROOT.SetBatch(True)
EPSILON_0 = 8.8541878128e-12
CM_TO_M = 1.0e-2
M_TO_CM = 100.0


Welcome to JupyROOT 6.30/06


## Configuration

In [4]:
# Input files
PAD_PLACEMENT_FILE = Path("../input/TPC_pad_placement.txt")
GAIN_MAP_FILE = Path("../input/dbb9e854-b14e-488e-9991-8f6f1962dffb.txt")
GAIN_METHOD = 1                         # 1 or 2
MULTIPLY_CHARGE_BY_GAIN = True          # requested convention
NORMALIZE_GAIN_WEIGHTED_TOTAL = True    # preserve total charge after introducing asymmetry

# Radial model
GENERATE_RADIAL_STUDY = True
FIT_GENERATED_CHARGE = True             # False -> use FIXED_ALPHA/FIXED_CONSTANT
GENERATION_MODE = "flat_eta"            # "flat_eta" or "isotropic_sphere"
N_GENERATED = 2_000_000
ETA_MAX = 1.2
Z_GEM_CM = 102.325
FIT_R_MIN_CM = 22.8132065669
FIT_R_MAX_CM = 75.9666751734
FIXED_ALPHA = np.array([2.0, 2.0, 2.0], dtype=float)
FIXED_CONSTANT = np.array([1.0, 1.0, 1.0], dtype=float)

# TPC electrostatic volume and GEM/module boundaries
A_CM, B_CM, L_CM = 20.0, 75.911, 102.325
# R1 includes the antenna-pad GEM area as requested.
MODULE_R_BOUNDS_CM = np.array([
    [22.8132065669, 40.0513196955],
    [41.6592025362 - 0.5*1.3179431693, 56.9695394315 + 0.5*1.3179431693],
    [58.9109633493 - 0.5*1.3175606263, 75.3666751734 + 0.5*1.3175606263],
])

# Source and observation grids. Increase NPHI only after a small validation run.
NR_SRC, NPHI_SRC, NZ_SRC = 36, 48, 40
NR_OBS, NPHI_OBS, NZ_OBS = 36, 48, 40
M_PHI_MAX = 24
N_RADIAL_MODES = 28
N_LONGITUDINAL_MODES = 28
OBS_BLOCK = 256
SRC_BLOCK = 512

RHO_REFERENCE_NC_PER_M3 = 20.0
K_EFF = 1.0
OUTPUT_ROOT = "sphenix_3d_ibf_field.root"
SAVE_CHARGE_MAP = True


## Parse real sector/module geometry and module gain map

In [5]:
def wrap_phi(phi):
    return (phi + np.pi) % (2*np.pi) - np.pi

def parse_pad_geometry(path):
    text = Path(path).read_text(errors="ignore")
    region = None
    data = {}
    for line in text.splitlines():
        m = re.search(r"region\s+(\d+)\s+first_layer", line)
        if m:
            region = int(m.group(1))
            continue
        m = re.search(r"side\s+(\d+)\s+sector\s+(\d+).*?first_phi\s+([-+0-9.eE]+).*?last_phi\s+([-+0-9.eE]+)", line)
        if m and region is not None and (region, int(m.group(1)), int(m.group(2))) not in data:
            side, sec = int(m.group(1)), int(m.group(2))
            first_phi, last_phi = float(m.group(3)), float(m.group(4))
            # Pad centers. Extend by half a pad pitch on both sides to get the active edge.
            delta = wrap_phi(last_phi - first_phi)
            npads = (94, 128, 192)[region]
            pitch = delta/(npads-1)
            edge0 = wrap_phi(first_phi - 0.5*pitch)
            edge1 = wrap_phi(last_phi + 0.5*pitch)
            data[(region, side, sec)] = (edge0, edge1)
    if len(data) != 3*2*12:
        raise RuntimeError(f"Expected 72 geometry entries, found {len(data)}")
    return data

def phi_in_interval(phi, edge0, edge1):
    width = (edge1-edge0) % (2*np.pi)
    return ((phi-edge0) % (2*np.pi)) <= width

def parse_gain_map(path, method=1):
    text = Path(path).read_text(errors="ignore")
    gain = np.ones((2, 12, 3), dtype=float)
    # Full-table columns end in Gain_m1 Gain_m2.
    pat = re.compile(r"^\s*([01])\s+(\d{1,2})\s+([123])\s+\d+\s+\d+\s+[-+0-9.eE]+\s+[yn]\s+[-+0-9.eE]+\s+[yn]\s+[-+0-9.eE]+\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)\s*$")
    found = 0
    for line in text.splitlines():
        m = pat.match(line)
        if not m: continue
        side, sec, mod = int(m.group(1)), int(m.group(2)), int(m.group(3))-1
        gain[side, sec, mod] = float(m.group(4 if method == 1 else 5))
        found += 1
    if found != 72:
        raise RuntimeError(f"Expected 72 gain entries, found {found}")
    return gain

pad_geometry = parse_pad_geometry(PAD_PLACEMENT_FILE)
gain_map = parse_gain_map(GAIN_MAP_FILE, GAIN_METHOD)
print("Module radial bounds [cm]:\n", MODULE_R_BOUNDS_CM)
print("Gain range:", gain_map.min(), gain_map.max(), "mean:", gain_map.mean())


Module radial bounds [cm]:
 [[22.81320657 40.0513197 ]
 [41.00023095 57.62851102]
 [58.25218304 76.02545549]]
Gain range: 0.801074 1.444409 mean: 1.0361793194444444


## Generate charge from spherical emission or flat pseudorapidity and optionally fit $C/r^\alpha$

In [ ]:
def generate_cylinder_radii(
    n,
    mode,
    z_plane_cm,
    eta_max,
    seed=12345,
):
    rng = np.random.default_rng(seed)

    if mode == "flat_eta":
        eta = rng.uniform(-eta_max, eta_max, n)

        # Avoid division by zero near eta = 0.
        eta = eta[np.abs(eta) > 1.0e-9]

        # Intersection with the plane z = +/-z_plane_cm.
        r = np.abs(z_plane_cm / np.sinh(eta))
        phi = rng.uniform(-np.pi, np.pi, len(r))

    elif mode == "isotropic_sphere":
        cos_theta = rng.uniform(-1.0, 1.0, n)
        theta = np.arccos(cos_theta)

        # Intersection with the plane z = +/-z_plane_cm.
        r = np.abs(z_plane_cm * np.tan(theta))
        phi = rng.uniform(-np.pi, np.pi, len(r))

    else:
        raise ValueError(
            f"Unknown GENERATION_MODE={mode!r}. "
            "Use 'flat_eta' or 'isotropic_sphere'."
        )

    keep = (
        np.isfinite(r)
        & (r >= FIT_R_MIN_CM)
        & (r <= FIT_R_MAX_CM)
    )

    return r[keep], phi[keep]


def power_law(r, c, alpha):
    return c / np.power(r, alpha)


def fit_radial_distribution(radii, nbins=70):
    edges = np.linspace(
        FIT_R_MIN_CM,
        FIT_R_MAX_CM,
        nbins + 1,
    )

    counts, _ = np.histogram(radii, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)

    # dN/dr
    y = counts / widths
    sigma = np.sqrt(np.maximum(counts, 1.0)) / widths

    use = counts > 10

    if np.count_nonzero(use) < 3:
        raise RuntimeError(
            "Too few populated radial bins for the power-law fit."
        )

    initial_alpha = 1.0
    initial_c = (
        y[use][0]
        * centers[use][0] ** initial_alpha
    )

    popt, pcov = optimize.curve_fit(
        power_law,
        centers[use],
        y[use],
        p0=(initial_c, initial_alpha),
        sigma=sigma[use],
        absolute_sigma=True,
        maxfev=20000,
    )

    return centers, y, sigma, popt, pcov


fitted_c = None
fitted_alpha = None
fitted_cov = None

if GENERATE_RADIAL_STUDY:
    generated_r, generated_phi = generate_cylinder_radii(
        n=N_GENERATED,
        mode=GENERATION_MODE,
        z_plane_cm=Z_GEM_CM,
        eta_max=ETA_MAX,
    )

    print(
        f"{GENERATION_MODE}: accepted "
        f"{len(generated_r)} / {N_GENERATED} generated particles"
    )

    centers, y, yerr, fitted, fitted_cov = (
        fit_radial_distribution(generated_r)
    )

    fitted_c, fitted_alpha = fitted
    alpha_error = np.sqrt(fitted_cov[1, 1])

    print(
        f"Fit dN/dr = C/r^alpha:\n"
        f"  C     = {fitted_c:.6g}\n"
        f"  alpha = {fitted_alpha:.6f} "
        f"+/- {alpha_error:.6f}"
    )

    plt.figure(figsize=(8, 5))

    plt.errorbar(
        centers,
        y,
        yerr=yerr,
        fmt=".",
        label="Generated cylinder crossings",
    )

    plt.plot(
        centers,
        power_law(centers, fitted_c, fitted_alpha),
        label=(
            rf"$C/r^{{\alpha}}$, "
            rf"$\alpha={fitted_alpha:.3f}$"
        ),
    )

    plt.yscale("log")
    plt.xlabel(r"$r$ [cm]")
    plt.ylabel(r"$dN/dr$")
    plt.title(
        f"Radial distribution: {GENERATION_MODE}"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


# Select the radial parameters used by the field calculation.
if FIT_GENERATED_CHARGE:
    if fitted_alpha is None or fitted_c is None:
        raise RuntimeError(
            "FIT_GENERATED_CHARGE=True requires "
            "GENERATE_RADIAL_STUDY=True."
        )

    RADIAL_ALPHA = np.full(
        3,
        fitted_alpha,
        dtype=float,
    )

    RADIAL_CONSTANT = np.full(
        3,
        fitted_c,
        dtype=float,
    )

    print("Using fitted radial charge parameters.")

else:
    RADIAL_ALPHA = FIXED_ALPHA.copy()
    RADIAL_CONSTANT = FIXED_CONSTANT.copy()

    print("Using fixed radial charge parameters.")


print("RADIAL_ALPHA    =", RADIAL_ALPHA)
print("RADIAL_CONSTANT =", RADIAL_CONSTANT)

flat_eta: accepted 155944; fitted alpha = 0.101933 +/- 0.074360


ValueError: 
C/r^{\\alpha}
     ^
ParseSyntaxException: Expected end_group, found '\'  (at char 5), (line:1, col:6)

<Figure size 800x500 with 1 Axes>

RADIAL_ALPHA = [0.10193342 0.10193342 0.10193342]
RADIAL_CONSTANT = [28739.09351585 28739.09351585 28739.09351585]


## Build signed-$z$ 3D source density using real GEM masks and gain multiplication

In [7]:
a, b, L = A_CM*CM_TO_M, B_CM*CM_TO_M, L_CM*CM_TO_M
r_src_edges = np.linspace(a, b, NR_SRC+1)
phi_src_edges = np.linspace(-np.pi, np.pi, NPHI_SRC+1)
z_src_edges = np.linspace(-L, L, NZ_SRC+1)
r_src = 0.5*(r_src_edges[:-1]+r_src_edges[1:])
phi_src = 0.5*(phi_src_edges[:-1]+phi_src_edges[1:])
z_src = 0.5*(z_src_edges[:-1]+z_src_edges[1:])

dr = np.diff(r_src_edges)[:,None,None]
dphi = np.diff(phi_src_edges)[None,:,None]
dz = np.diff(z_src_edges)[None,None,:]
dV = r_src[:,None,None]*dr*dphi*dz

rho_shape = np.zeros((NR_SRC,NPHI_SRC,NZ_SRC), dtype=np.float64)
cell_gain = np.ones_like(rho_shape)
cell_module = -np.ones_like(rho_shape, dtype=np.int8)

for ir, r_m in enumerate(r_src):
    r_cm = r_m*M_TO_CM
    module = next((m for m,(lo,hi) in enumerate(MODULE_R_BOUNDS_CM) if lo <= r_cm <= hi), None)
    if module is None: continue
    radial = RADIAL_CONSTANT[module] / r_cm**RADIAL_ALPHA[module]
    for ip, phi in enumerate(phi_src):
        for iz, z_m in enumerate(z_src):
            side = 0 if z_m > 0 else 1
            sector = next((s for s in range(12) if phi_in_interval(phi, *pad_geometry[(module,side,s)])), None)
            if sector is None: continue
            g = gain_map[side,sector,module] if MULTIPLY_CHARGE_BY_GAIN else 1.0
            rho_shape[ir,ip,iz] = radial*g
            cell_gain[ir,ip,iz] = g
            cell_module[ir,ip,iz] = module

active = rho_shape > 0
if not np.any(active): raise RuntimeError("No active GEM/source cells")
if NORMALIZE_GAIN_WEIGHTED_TOTAL and MULTIPLY_CHARGE_BY_GAIN:
    # Remove only the overall gain-induced normalization while preserving the r model and asymmetry.
    base = np.zeros_like(rho_shape)
    base[active] = rho_shape[active]/cell_gain[active]
    norm = np.sum(base*dV)/np.sum(rho_shape*dV)
    rho_shape *= norm

target_rho = K_EFF*RHO_REFERENCE_NC_PER_M3*1e-9
rho = target_rho*rho_shape/(np.sum(rho_shape*dV)/np.sum(dV[active]))
charge = rho*dV
print("Active fraction:", active.mean())
print("Total charge [C]:", charge.sum())


Active fraction: 0.8888888888888888
Total charge [C]: 6.327714118834932e-08


## Modal Green-function preparation

In [8]:
def F(k,m):
    return special.jv(m,k*b)*special.yv(m,k*a)-special.yv(m,k*b)*special.jv(m,k*a)
def Rfun(k,m,r):
    return special.jv(m,k*r)*special.yv(m,k*a)-special.yv(m,k*r)*special.jv(m,k*a)
def dRfun(k,m,r):
    return k*(special.jvp(m,k*r,1)*special.yv(m,k*a)-special.yvp(m,k*r,1)*special.jv(m,k*a))
def roots_for_m(m,nroot):
    roots=[]; step=0.2*np.pi/(b-a); x=max(1e-6,0.1*np.pi/(b-a)); fx=F(x,m)
    xmax=(nroot+m+30)*np.pi/(b-a)
    while x+step <= xmax and len(roots)<nroot:
        y=x+step; fy=F(y,m)
        if np.isfinite(fx) and np.isfinite(fy) and fx*fy<0:
            rt=optimize.brentq(F,x,y,args=(m,))
            if not roots or abs(rt-roots[-1])>1e-9: roots.append(rt)
        x,fx=y,fy
    if len(roots)!=nroot: raise RuntimeError(f"Only {len(roots)} roots for m={m}")
    return np.asarray(roots)

def radial_norm(k,m):
    val,_=integrate.quad(lambda rr: rr*Rfun(k,m,rr)**2, a, b, epsabs=1e-10, epsrel=1e-8, limit=200)
    return val

k_modes={}; radial_norms={}
for m in range(M_PHI_MAX+1):
    k_modes[m]=roots_for_m(m,N_RADIAL_MODES)
    radial_norms[m]=np.array([radial_norm(k,m) for k in k_modes[m]])
q_modes=np.arange(1,N_LONGITUDINAL_MODES+1)*np.pi/(2*L)


## Memory-bounded convolution (kernels are never stored)

In [9]:
r_obs_edges=np.linspace(a,b,NR_OBS+1)
phi_obs_edges=np.linspace(-np.pi,np.pi,NPHI_OBS+1)
z_obs_edges=np.linspace(-L,L,NZ_OBS+1)
r_obs=0.5*(r_obs_edges[:-1]+r_obs_edges[1:])
phi_obs=0.5*(phi_obs_edges[:-1]+phi_obs_edges[1:])
z_obs=0.5*(z_obs_edges[:-1]+z_obs_edges[1:])

Robs,Pobs,Zobs=np.meshgrid(r_obs,phi_obs,z_obs,indexing='ij')
Rsrc,Psrc,Zsrc=np.meshgrid(r_src,phi_src,z_src,indexing='ij')
obs=np.column_stack([Robs.ravel(),Pobs.ravel(),Zobs.ravel()])
src=np.column_stack([Rsrc.ravel(),Psrc.ravel(),Zsrc.ravel()])
qsrc=charge.ravel()
keep=qsrc!=0
src=src[keep]; qsrc=qsrc[keep]

Ex=np.zeros(len(obs)); Ey=np.zeros(len(obs)); Ez=np.zeros(len(obs)); Phi=np.zeros(len(obs))
start=time.time()
# Dirichlet boundaries at z=-L,+L use u=z+L in [0,2L].
for o0 in range(0,len(obs),OBS_BLOCK):
    ob=obs[o0:o0+OBS_BLOCK]; ro,po,zo=ob.T; uo=zo+L
    e_r=np.zeros(len(ob)); e_p=np.zeros(len(ob)); e_z=np.zeros(len(ob)); pot=np.zeros(len(ob))
    for s0 in range(0,len(src),SRC_BLOCK):
        sb=src[s0:s0+SRC_BLOCK]; qs=qsrc[s0:s0+SRC_BLOCK]; rs,ps,zs=sb.T; us=zs+L
        dph=po[:,None]-ps[None,:]
        for m in range(M_PHI_MAX+1):
            pref=(2.0 if m>0 else 1.0)/(2*np.pi)
            c=np.cos(m*dph); sn=np.sin(m*dph)
            for k,norm in zip(k_modes[m],radial_norms[m]):
                rr_o=Rfun(k,m,ro); drr_o=dRfun(k,m,ro); rr_s=Rfun(k,m,rs)
                radial=np.outer(rr_o,rr_s)/norm
                radial_r=-np.outer(drr_o,rr_s)/norm
                radial_p=(m/ro[:,None])*radial if m>0 else 0.0
                for q in q_modes:
                    den=k*k+q*q
                    zpart=(1/L)*np.outer(np.sin(q*uo),np.sin(q*us))/den
                    zder=-(1/L)*np.outer(q*np.cos(q*uo),np.sin(q*us))/den
                    common=pref*c*zpart
                    pot += (radial*common)@qs/EPSILON_0
                    e_r += (radial_r*common)@qs/EPSILON_0
                    if m>0: e_p += (radial_p*(pref*sn)*zpart)@qs/EPSILON_0
                    e_z += (radial*(pref*c)*zder)@qs/EPSILON_0
    Ex[o0:o0+len(ob)] = e_r*np.cos(po)-e_p*np.sin(po)
    Ey[o0:o0+len(ob)] = e_r*np.sin(po)+e_p*np.cos(po)
    Ez[o0:o0+len(ob)] = e_z
    Phi[o0:o0+len(ob)] = pot
    print(f"observation block {o0}:{o0+len(ob)} elapsed {time.time()-start:.1f} s")

shape=(NR_OBS,NPHI_OBS,NZ_OBS)
Ex=Ex.reshape(shape); Ey=Ey.reshape(shape); Ez=Ez.reshape(shape); Phi=Phi.reshape(shape)
print("max fields [V/m]",np.max(np.abs(Ex)),np.max(np.abs(Ey)),np.max(np.abs(Ez)))


KeyboardInterrupt: 

## Write PHGarfield-compatible TH3 maps

In [ ]:
def make_th3(name,title,arr,xedges,yedges,zedges):
    h=ROOT.TH3F(name,title,len(xedges)-1,np.asarray(xedges,dtype='d'),len(yedges)-1,np.asarray(yedges,dtype='d'),len(zedges)-1,np.asarray(zedges,dtype='d'))
    h.SetDirectory(0)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            for k in range(arr.shape[2]): h.SetBinContent(i+1,j+1,k+1,float(arr[i,j,k]))
    return h

f=ROOT.TFile(OUTPUT_ROOT,"RECREATE")
d=f.mkdir("Field3D"); d.cd()
for h in [
    make_th3("hEx","local Ex correction; r [cm]; phi [rad]; z [cm]; Ex [V/m]",Ex,r_obs_edges*M_TO_CM,phi_obs_edges,z_obs_edges*M_TO_CM),
    make_th3("hEy","local Ey correction; r [cm]; phi [rad]; z [cm]; Ey [V/m]",Ey,r_obs_edges*M_TO_CM,phi_obs_edges,z_obs_edges*M_TO_CM),
    make_th3("hEz","local Ez correction; r [cm]; phi [rad]; z [cm]; Ez [V/m]",Ez,r_obs_edges*M_TO_CM,phi_obs_edges,z_obs_edges*M_TO_CM),
]: h.Write()
if SAVE_CHARGE_MAP:
    f.cd(); qd=f.mkdir("Charge3D"); qd.cd()
    make_th3("hRho","rho; r [cm]; phi [rad]; z [cm]; rho [C/m3]",rho,r_src_edges*M_TO_CM,phi_src_edges,z_src_edges*M_TO_CM).Write()
f.cd()
ROOT.TNamed("coordinate_convention","local cylindrical axes: phi=atan2(y,x); signed z; side0=z>0; side1=z<0; field bins V/m").Write()
ROOT.TNamed("gain_method",str(GAIN_METHOD)).Write()
ROOT.TNamed("fit_generated_charge",str(int(FIT_GENERATED_CHARGE))).Write()
f.Close()
print("Wrote",OUTPUT_ROOT)
